# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bajwaycodes/Kashif-Working-Repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import pandas as pd, numpy as np
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
# Signal 1: staleness vs declining rate
df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, df["days_since_last_update"].max()],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)

stale_table = df.groupby("stale_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    declining_rate=("is_declining_label", "mean")
)
print(stale_table)
print("\nOverall declining rate (base rate):", round(df["is_declining_label"].mean(), 3))

                  n  declining_rate
stale_bucket                       
<90d          20655        0.512031
90-180d        9171        0.611057
180-365d        169        0.467456
365d+             5        0.600000

Overall declining rate (base rate): 0.542


**Verdict on Signal 1 (staleness): MIXED**

Declining rate does not rise monotonically with staleness. It rises from <90d (0.512)
to 90-180d (0.611) — a real, well-supported jump (n=9,171) above the 0.542 base rate.
But it then *drops* for 180-365d (0.467, n=169), below the base rate, and the 365d+
bucket (0.600) has only 5 rows — not enough to trust.

Honest reading: staleness isn't a clean linear signal here. The 90-180 day window
does carry real elevated risk, which is enough to justify using "stale ≥ 180 days"
loosely as one ingredient in a rule, but it does not on its own justify believing
"older = worse" without limit — a page stale for 300+ days is not more at-risk than
one stale for 100 days in this data, contrary to what a naive staleness-only rule
would assume.

In [9]:
# Signal 2: CTR vs position, controlling for position tier
df_visible = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 100)].copy()

df_visible["ctr_bucket"] = pd.qcut(df_visible["ctr"], q=3, labels=["low_ctr", "mid_ctr", "high_ctr"], duplicates="drop")

ctr_table = df_visible.groupby("ctr_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    declining_rate=("is_declining_label", "mean")
)
print(ctr_table)
print("\nBase rate among visible pages (position 1-20):", round(df_visible["is_declining_label"].mean(), 3))

               n  declining_rate
ctr_bucket                      
low_ctr     5271        0.716942
mid_ctr     4886        0.603766
high_ctr    4934        0.530604

Base rate among visible pages (position 1-20): 0.619


               n  declining_rate
ctr_bucket                      
low_ctr     5271        0.716942
mid_ctr     4886        0.603766
high_ctr    4934        0.530604

Base rate among visible pages (position 1-20): 0.619

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import os

os.makedirs("work/outputs", exist_ok=True)

stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = stale * visible * df["impressions_90d"]

ctr_median = df["ctr"].median()

def assign_reason_code(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    elif 0 < row["avg_position"] <= 20 and row["ctr"] < ctr_median:
        return "low_ctr_visible_page"
    elif row["trend_direction"] == "down" and row["impressions_90d"] >= 100:
        return "declining_with_demand"
    else:
        return "no_strong_signal"

df["reason_code"] = df.apply(assign_reason_code, axis=1)
df["action"] = np.where(df["baseline_score"] > 0, "review_for_refresh", "monitor")

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "client_id", "baseline_score", "reason_code", "action",
     "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]
]

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} ranked rows to work/outputs/baseline_action_score.csv")
queue.head(10)

Wrote 30000 ranked rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,61678,194,19.7,0.15,down
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,59472,194,24.8,0.13,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,25715,194,22.2,0.23,down
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,13299,193,10.5,0.49,down
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,7812,194,39.0,0.01,down
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,7558,193,17.9,0.20,down
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,4590,194,31.0,0.00,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,4556,194,16.4,0.33,down
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,4429,194,25.3,0.38,down
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,1697,193,15.8,0.12,down


In [11]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values
base_rate = y.mean()

for k in (20, 50):
    p = precision_at_k(df["baseline_score"], y, k)
    print(f"Precision@{k}: {p:.3f}   (base rate: {base_rate:.3f})")

import json
metrics = {
    "base_rate": float(base_rate),
    "precision_at_20": float(precision_at_k(df["baseline_score"], y, 20)),
    "precision_at_50": float(precision_at_k(df["baseline_score"], y, 50)),
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

Precision@20: 0.900   (base rate: 0.542)
Precision@50: 0.680   (base rate: 0.542)
{'base_rate': 0.5420666666666667, 'precision_at_20': 0.9, 'precision_at_50': 0.68}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. **content_cf56e2e2e282** — action: review_for_refresh. Why: stale (194d),
   61,678 impressions, position 19.7, low CTR (0.15), trending down. Would be
   wrong if: this page's high impression count is driven by one seasonal spike
   rather than sustained demand — score doesn't distinguish steady traffic from
   a one-time surge.

2. **content_7368877ea310** — action: review_for_refresh. Why: stale (194d),
   59,472 impressions, position 24.8, CTR 0.13, trending down. Would be wrong
   if: position 24.8 means this page isn't realistically reachable on page 1
   anyway — refreshing content won't fix a position problem this far down.

3. **content_1bfaa38ff26c** — action: review_for_refresh. Why: stale (194d),
   25,715 impressions, CTR 0.23 (better than rows above), trending down. Would
   be wrong if: the CTR here is actually reasonable for position 22.2, meaning
   this page isn't under-performing its position — it just has lower demand.

4. **content_0a91db491d14** — action: review_for_refresh. Why: stale (193d),
   13,299 impressions, position 10.5 (near page-1 boundary), CTR 0.49 (much
   higher, near median). Would be wrong if: with CTR this close to the median,
   this page may not have a real problem at all — it might just be aging
   naturally, not declining.

5. **content_5feee3994adb** — action: review_for_refresh. Why: stale (194d),
   7,812 impressions, position 39.0, CTR 0.01 (near-zero). Would be wrong if:
   position 39 is so far down that near-zero CTR is expected for any page
   there — this isn't a CTR problem, it's a ranking problem the content team
   can't fix by editing the page.

6. **content_c2d929d83eaa** — action: review_for_refresh. Why: stale (193d),
   7,558 impressions, position 17.9, CTR 0.20. Would be wrong if: this page is
   only in the queue because of its raw impression volume, not because
   anything about it looks genuinely broken relative to peers at that position.

7. **content_b16bd7307b39** — action: review_for_refresh. Why: stale (194d),
   4,590 impressions, position 31.0, CTR 0.00 (literally zero clicks). Would
   be wrong if: zero CTR here just reflects position 31 being effectively
   invisible — not a content quality issue.

8. **content_fe16a55cd13d** — action: review_for_refresh. Why: stale (194d),
   4,556 impressions, position 16.4, CTR 0.33 (closer to healthy). Would be
   wrong if: CTR 0.33 at position 16 might actually be reasonable — this page
   may be flagged mainly because it's stale, not because it's truly declining.

9. **content_ecb6215e79fd** — action: review_for_refresh. Why: stale (194d),
   4,429 impressions, position 25.3, CTR 0.38. Would be wrong if: similar to
   row 8 — CTR looks acceptable for its position, so "stale" alone may be
   doing most of the work in flagging this page, not real evidence of decline.

10. **content_928af3e22c80** — action: review_for_refresh. Why: stale (193d),
    1,697 impressions (lowest in top 10), position 15.8, CTR 0.12. Would be
    wrong if: with impressions this low relative to the rest of the top 10,
    this page barely clears the visibility threshold — it's here mostly
    because it's stale and marginally visible, not because it's a strong
    priority candidate.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:**

The clearest weakness isn't any single row — it's that all 10 of the top-10 rows
belong to the same client (`client_7f2253d7e2`), with nearly identical staleness
(193-194 days). That's a sign the score (`stale × visible × impressions_90d`) is
dominated by raw impression volume, so any client with unusually high-traffic pages
will flood the top of the queue, regardless of whether their pages are more
genuinely at-risk than pages from smaller clients.

Two specific rows illustrate the deeper problem: rows 8 and 9
(content_fe16a55cd13d, content_ecb6215e79fd) have CTR of 0.33 and 0.38 at
reasonably good positions (16.4 and 25.3) — those CTR values don't look obviously
broken. They only made the top 10 because they're stale and have enough impressions
to clear the visibility bar. That means "stale" is doing most of the ranking work
for these two, not real evidence of decline — which is consistent with the MIXED
verdict from Section 1: staleness alone is not a clean, reliable signal.

This means Precision@20 = 0.900 is real, but partly inflated by one high-volume
client's pages, which are easy to rank correctly because of scale, not because the
rule is precisely targeting the riskiest pages across the whole dataset. A version
of this rule that normalizes for client size, or caps how many pages one client can
contribute to the top of the queue, would likely be a fairer test of the rule's
actual quality.

**Leakage check:**

The score uses only `days_since_last_update`, `impressions_90d`, and implicitly the
CTR/position fields through the reason-code logic — all pre-decision observed
signals. `trend_direction` is used only to build the *evaluation label*
(`is_declining_label`), never as an input to `baseline_score` itself — confirmed by
inspecting the score formula, which references only `stale`, `visible`, and
`impressions_90d`. No FlyRank product flags (`health_score`, `priority_score`,
`action_type`) exist in this dataset, so there's nothing of that kind to leak in.
No future-window data was used — everything comes from the same trailing-90-day
snapshot.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.